# ⚖️ Lexior : Distillation CoT de LLaMA 3 pour Spécialisation Juridique

Ce notebook vous guide pas à pas dans le processus de **distillation de Chain-of-Thought (CoT)** pour spécialiser un modèle local (ex. LLaMA 3.1 8B ou Qwen 2.5 7B) dans le raisonnement juridique canadien et québécois selon la méthode **IRAC** (Issue, Rule, Application, Conclusion).

Le modèle apprendra à générer sa réflexion dans un bloc `<thinking>...</thinking>` avant de donner sa conclusion juridique finale, de générer des sorties au format JSON structuré ou d'effectuer des appels de fonction (Tool Calling) compatibles avec **LexiorNotebook**.

## 🛠️ Étape 1 : Préparation de l'environnement

Commençons par installer les bibliothèques indispensables. Si vous êtes sur RunPod, PyTorch et CUDA sont déjà présents.

In [ ]:
# 1. Vérification du GPU
!nvidia-smi

In [ ]:
# 2. Installation des dépendances
!pip install -r requirements.txt

# 3. Installation d'Unsloth (optimisé pour GPU NVIDIA)
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothdev/unsloth.git"

In [ ]:
# 4. Authentification Hugging Face et Weights & Biases (Optionnel)
import os
from huggingface_hub import notebook_login

# Token Hugging Face
if "HF_TOKEN" in os.environ:
    from huggingface_hub import HfFolder
    HfFolder.save_token(os.environ["HF_TOKEN"])
    print("Token Hugging Face chargé depuis l'environnement !")
else:
    print("Pour téléverser vos modèles sur Hugging Face, connectez-vous :")
    notebook_login()

# Weights & Biases (Suivi des hyperparamètres et métriques)
import wandb
if "WANDB_API_KEY" in os.environ:
    wandb.login(key=os.environ["WANDB_API_KEY"])
    print("Connecté à Weights & Biases via la clé d'environnement WANDB_API_KEY !")
else:
    print("\nSi vous souhaitez suivre l'entraînement sur Weights & Biases, lancez : wandb.login()")

## 📦 Étape 2 : Chargement et exploration du Dataset

Nous utilisons le dataset `SuperMust/irac-thinking` qui contient des exemples structurés de raisonnement juridique avec un champ séparé `thinking` pour le CoT.

In [ ]:
from datasets import load_dataset

dataset_name = "SuperMust/irac-thinking"
print(f"Téléchargement du dataset : {dataset_name}...")
raw_dataset = load_dataset(dataset_name, split="train")

print(f"Nombre d'exemples : {len(raw_dataset)}")
print("Aperçu du premier exemple brut :")
import pprint
pprint.pprint(raw_dataset[0])

## ✍️ Étape 3 : Formatage du Dataset pour la distillation CoT

Nous devons fusionner le raisonnement `thinking` et la conclusion `content` de l'assistant juridique dans le format suivant :
```text
<thinking>
[Raisonnement juridique IRAC en droit canadien/québécois]
</thinking>

[Conclusion finale ou structure JSON/Tool Call]
```
Nous forçons l'alignement sur la juridiction québécoise/canadienne et le respect strict du format de citation de LexiorGPT.

In [ ]:
from transformers import AutoTokenizer

# Chargement du tokenizer Llama 3
model_name = "unsloth/llama-3-8b-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_cot_dataset(example):
    messages_bruts = example.get("messages", [])
    messages_formates = []
    
    for msg in messages_bruts:
        role = msg["role"]
        if role == "developer":
            role = "system"
            
        content = msg["content"]
        thinking = msg.get("thinking", "")
        
        # Aligner le prompt système sur le droit canadien/québécois et les citations de bas de page Lexior
        if role == "system":
            content = (
                "Tu es un assistant juridique Lexior, spécialisé en droit canadien et québécois. "
                "Raisonne en français selon le format IRAC. Tu dois obligatoirement baser tes analyses "
                "sur la législation et la jurisprudence canadienne/québécoise (ex: Code civil du Québec, CanLII). "
                "Lorsque tu as fini de raisonner dans tes balises <thinking>, formate tes citations de bas de page "
                "strictement sous la forme [^1]:{\"type\":\"url\",\"url\":\"https://www.canlii.org/...\",\"title\":\"Titre\"}."
            )
            
        if role == "assistant":
            if thinking:
                # Injection des balises <thinking> de CoT
                content_final = f"<thinking>\n{thinking}\n</thinking>\n\n{content}"
            else:
                content_final = content
            messages_formates.append({"role": role, "content": content_final})
        else:
            messages_formates.append({"role": role, "content": content})
            
    # Application du chat template
    text = tokenizer.apply_chat_template(
        messages_formates, 
        tokenize=False, 
        add_generation_prompt=False
    )
    
    return {"text": text}

# Formatage
formatted_dataset = raw_dataset.map(
    format_cot_dataset, 
    remove_columns=raw_dataset.column_names
)

print("Exemple de texte formaté final envoyé à l'entraînement :")
print("-" * 80)
print(formatted_dataset[0]["text"])
print("-" * 80)

## 🚀 Étape 4 : Initialisation du modèle avec Unsloth

Nous chargeons le modèle en mode 4-bit (pour économiser la mémoire GPU) et lui appliquons les adaptateurs LoRA.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096

print("Chargement du modèle 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Vous pouvez aussi utiliser "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
    max_seq_length = max_seq_length,
    dtype = None, # Détection auto
    load_in_4bit = True
)

print("Configuration des adaptateurs LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)

## 🏋️ Étape 5 : Configuration des expérimentations et Lancement du Fine-Tuning

Vous pouvez choisir d'activer le tracking sur **Weights & Biases** ou d'utiliser un dossier de logs **TensorBoard** pour comparer vos différentes expérimentations juridiques (par exemple, en faisant varier le learning rate ou le rang de LoRA).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Split Train / Validation rapide (95% / 5%)
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)

# Configuration du type de logger pour cette expérimentation
# Options : "wandb", "tensorboard" ou "none"
report_to_logger = "wandb" if "WANDB_API_KEY" in os.environ else "tensorboard"
nom_experimentation = "llama3-canadian-legal-r16"

print(f"Suivi de l'expérience activé via : {report_to_logger} (Nom : {nom_experimentation})")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        save_steps = 100,
        evaluation_strategy = "steps",
        eval_steps = 100,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs/checkpoints",
        report_to = report_to_logger,
        run_name = nom_experimentation if report_to_logger != "none" else None,
        logging_dir = "outputs/logs" # Utile si vous utilisez TensorBoard en local
    )
)

In [ ]:
print("Début de l'entraînement...")
trainer_stats = trainer.train()
print("Entraînement terminé !")

## 🧪 Étape 6 : Test local du modèle (Inférence CoT et appels d'outils)

Vérifions le comportement du modèle avec un cas juridique pratique. Nous voulons voir s'il génère bien la balise `<thinking>`, respecte le format de raisonnement IRAC du droit québécois et formate l'appel d'outil CanLII `search` attendu par le backend de **LexiorNotebook**.

In [ ]:
# Activation du mode d'inférence rapide d'Unsloth
FastLanguageModel.for_inference(model)

# Exemple d'inférence combinant CoT et Tool Calling (Format Regex fallback de LexiorNotebook)
messages = [
    {
        "role": "system", 
        "content": "Tu es un assistant juridique Lexior, spécialisé en droit canadien et québécois. Raisonne en français selon le format IRAC. Tu as accès à l'outil 'search' (CanLII). Format de retour : search {\"query\": \"mots-clés\", \"language\": \"fr\"}"
    },
    {
        "role": "user", 
        "content": "Quels critères régissent la garde partagée des enfants en cas de séparation des parents au Québec ? Recherche la jurisprudence pertinente."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

print("Génération de la réponse...")
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.2,
    top_p = 0.95
)

reponse = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("\n=== Réponse Générée (Réflexion + Tool Calling) ===\n")
print(reponse)

## 💾 Étape 7 : Sauvegarde et Export (LoRA, complet, GGUF)

Nous pouvons maintenant sauvegarder les adaptateurs ou exporter le modèle.

In [ ]:
# 1. Sauvegarde des adapters LoRA uniquement (très léger)
model.save_pretrained("outputs/final_model/lora_adapters")
tokenizer.save_pretrained("outputs/final_model/lora_adapters")
print("Adapters LoRA sauvegardés localement !")

In [ ]:
# 2. Sauvegarde du modèle complet fusionné 16-bit
# model.save_pretrained_merged("outputs/final_model/merged_16bit", tokenizer, save_method = "merged_16bit")
# print("Modèle fusionné FP16 sauvegardé !")

In [ ]:
# 3. Export direct au format GGUF (prêt pour Ollama / Lexior)
# La méthode q4_k_m est recommandée pour garder un excellent équilibre qualité/vitesse.
print("Exportation GGUF en cours (cela peut prendre quelques minutes)...")
model.save_pretrained_gguf(
    "outputs/final_model/gguf_q4_k_m", 
    tokenizer, 
    quantization_method = "q4_k_m"
)
print("Export GGUF terminé avec succès dans outputs/final_model/gguf_q4_k_m !")

In [ ]:
# 4. Optionnel : Pousser vers Hugging Face Hub
# hf_repo_id = "votre_username/nom-du-modele"
# model.push_to_hub_gguf(f"{hf_repo_id}-gguf", tokenizer, quantization_method = "q4_k_m")
# print(f"Modèle poussé sur HF Hub : {hf_repo_id}-gguf")